# Measuring the environemental impacts of deep learning

Constance Douwes - Adapted from Lucia Bouza and Aurélie Bugeau

# Training costs

In this part, we will try to compute the energy consumption of training deep learning models for classification tasks alond with the associated carbon emissions.

## Using Green Algorithms methodology

With green algorithm, the energy consumption is computed as follows:
$$E = T \times \rm{PUE} \times \left(\sum_{c\in {cores}} P_{c}\times \rm{usage}_c + P_{memory} \right) \tag{1}$$
where
- $T$ is the running time in hours
- $\rm{PUE}$ is the Power Usage Effectiveness. It is an efficiency coefficient of the data center.
- $P_{memory}$ is the power consumed by the memory in Watt.
- $cores$ is the set of all CPU and GPU cores
- $P_{c}$ is the power consumed by core $c$ (CPU or GPU) in Watt.
- $\rm{usage}_c$ is the usage factor of core $c$, between $0$ and $1$.


The *carbon footprint* is calculated according to the following formula:

$$CO_2eq = E \times CI \tag{2}$$
where
- $CI$ is the carbon intensity factor of the region where the code is being run

In the following, we will try to find the appropriate information depending on the hardware and infrastructure used.


### CPU

To know the number of available CPUs and the model on the machine where you are running your program, you can execute the following commands.

- **Linux**: `cat /proc/cpuinfo` will display information about the available CPUs. To make the information easy to read, we use the following script that determines the number of configured physical and logical CPUs. You can also use `lscpu`.
- **MacOS**: `sysctl -n machdep.cpu.brand_string` will display Chip Brand, Processor Type, Chip Model and CPU Speed. Detailed info on cpu can further be obtained with the command `sysctl -a | grep machdep.cpu`. Other option can be browsing: **System Settings > General > About > Processor.**
- **Windows**: `wmic cpu get name, numberofcores` will display same information as MacOS command. Other option can be browsing: **Task Manager > Performance > CPU.**

In [ ]:
!cat /proc/cpuinfo | \
awk -v FS=':' '                                       \
  /^physical id/ { if(nb_cpu<$2)  { nb_cpu=$2 } }     \
  /^cpu cores/   { if(nb_cores<$2){ nb_cores=$2 } }   \
  /^processor/   { if(nb_units<$2){ nb_units=$2 } }   \
  /^model name/  { model=$2 }                         \
                                                      \
  END{                                                \
   nb_cpu=(nb_cpu+1);                                 \
   nb_units=(nb_units+1);                             \
                                                      \
   print "CPU model:",model;                          \
   print nb_cpu,"CPU,",nb_cores,"physical cores per CPU, total",nb_units,"logical CPU units" \
 }'

### GPU

Knowing the number and model of GPUs, can be done in the following ways:

- **Linux**: `lshw -C display`
- **MacOS**: browse: **System Settings > General > About > Graphics**
- **Windows**: browse: **Task Manager > Performance > GPU**


> Remark: The virtual machine used by Colab does not have some Linux commands. We know that the GPUs used by Colab are Nvidia, so we can use the nvidia-specific command:`!nvidia-smi -L`. This will show the number of available GPUs and their model and UUID.

In [ ]:
!nvidia-smi -L

### Compute energy from cores

To compute $P_c$ , Green Algorithms use the Thermal Design Power (**TDP**) of the model of the processing unit provided by the manufacturer. A core power usage is assumed to be equal to the TDP divided by the number of cores (if a chip has 2 cores and a TDP of 50W, then the TDP per core is 25).

Use online documentations of CPU and GPU to find TDP values of your CPU and GPU, compute the value of $\sum_{c\in {cores}} P_{c}\times \rm{usage}_c$ in Watt, considering $\rm{usage}_c =1$.

In [ ]:
TDP_GPU = #W
TDP_CPU = #W
P_cores = #W
print(f"Total power cores : {P_cores}")

### Memory



We can check the amount of available memory available for the GPU using the following commands:

- **Linux**: `grep MemTotal /proc/meminfo`. The command returns the value in KB. It will be necessary to convert it to GB to use the tool (1 GB = 1024 MB = 1048576 KB). Another command is `free -h --si | awk  '/Mem:/{print $2}'`
- **MacOS**: `system_profiler SPHardwareDataType | grep "Memory:"`. Other option  can be browsing: **System Settings > General > About > Memory.**
- **Windows**: `systeminfo | findstr /C:"Total Physical Memory"`. The command returns the value in MB, it will be necessary to convert it to GB to use the tool. Other option  can be browsing: **Task Manager > Performance > Memory**


Green Algorithms, considers a consumption of 0.3725 W per GB of memory available (if we have all the server memory available, it will account for all the server memory. If we are in an HPC cluster, it will account only for the amount of memory requested, regardless of how much the process consumes). The value 0.3725 was obtained experimentally <a name="cite_ref-2"></a>[[2]](#cite_note-2).

In [ ]:
!grep MemTotal /proc/meminfo

13286936/1048576

In [ ]:
Memory_kB = #
Memory_GB = #
P_mem = #
print(f"Total power memory : {P_mem}")

### Location

We can check the execution location with the command:
- **Linux, Windows and MacOS**: `curl ipinfo.io`

> **Remark:** In the case of Colab, you cannot choose where to execute the codes. Nevertheless, you can also use previous command and then determine the datacenter where the code is executed with the following [link](https://cloud.google.com/about/locations?hl=es). We do not know in advance in which datacenter the virtual machine where the notebook will run will be created. It is necessary to execute the command every time we create or reset the runtime, since the location may change.

In [ ]:
!curl ipinfo.io

Check the carbon intensity factor from this location at : https://app.electricitymaps.com/map/fifteen_minutes

In [ ]:
CI = #gC02/kWh

### Time to train a network

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import psutil
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torchsummary import summary


In [ ]:
# Load MNIST dataset
train_data = MNIST(root='./data', train=True, transform=ToTensor(), download=True)
test_data = MNIST(root='./data', train=False, transform=ToTensor(), download=True)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# Define the model
class Digits(nn.Module):
    def __init__(self):
        super(Digits, self).__init__()
        self.flatten = nn.Flatten(start_dim=1)
        self.hidden_layer = nn.Linear(28*28, 32)
        self.output_layer = nn.Linear(32, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.sigmoid(self.hidden_layer(x))
        x = self.output_layer(x)
        return x


In [ ]:
# Define device where to run the training
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = Digits().to(device)
# model = Digits_CNN().to(device)

In [ ]:
print('Using gpu: %s ' % torch.cuda.is_available())

Using gpu: True 


In [ ]:
# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train(model, nb_epochs):
  for epoch in range(nb_epochs):
      train_loss = 0.0
      train_acc = 0.0
      val_loss = 0.0
      val_acc = 0.0

      model.train()
      for i, (inputs, labels) in enumerate(train_loader):
          inputs, labels = inputs.to(device), labels.to(device)
          optimizer.zero_grad()
          outputs = model(inputs)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          train_loss += loss.item() * inputs.size(0)
          _, preds = torch.max(outputs, 1)
          train_acc += torch.sum(preds == labels.data)

      model.eval()
      with torch.no_grad():
          for i, (inputs, labels) in enumerate(test_loader):
              inputs, labels = inputs.to(device), labels.to(device)
              outputs = model(inputs)
              loss = criterion(outputs, labels)

              val_loss += loss.item() * inputs.size(0)
              _, preds = torch.max(outputs, 1)
              val_acc += torch.sum(preds == labels.data)

      epoch_train_loss = train_loss / len(train_data)
      epoch_train_acc = train_acc / len(train_data)
      epoch_val_loss = val_loss / len(test_data)
      epoch_val_acc = val_acc / len(test_data)

      print('Epoch [{}/{}], Train Loss: {:.4f}, Train Acc: {:.4f}, Val Loss: {:.4f}, Val Acc: {:.4f}'
            .format(epoch+1, nb_epochs, epoch_train_loss, epoch_train_acc, epoch_val_loss, epoch_val_acc))

We will use `time` libreary to measure the time of a piece of code.
More info here : https://docs.python.org/3/library/time.html#time.time


In [ ]:
import time
start = time.time()

train(model, 1) #you can change the number of epopchs if needed

stop = time.time()
print('Real Time: {:.4f} seconds'.format(stop-start))

In [ ]:
time_s = # sesconds
time_h = # hours

Watts

In [ ]:
Energy = #Wh
CO2e =  #gCO2
print(f"Energy : {Energy:.2f} Wh")
print(f"Emissions : {CO2e:.2f} gCO2")

Compare your results with the online tool Green Algorithm :  https://calculator.green-algorithms.org/


## Using CodeCarbon

In this section, we want to monitor closely the consumption using `codecarbon` library.


In [ ]:
import psutil

In [ ]:
!pip install codecarbon > /dev/null

There are several ways of using the library, but here we use the recommended way for notebooks, using the `start` and `stop` functions of the tracker. Other options can be found in the [documentation](https://docs.codecarbon.io/latest/tutorials/first-tracking/#step-2-track-emissions-from-a-computation).
 The package by default logs data into a CSV file named `emissions.csv` in the current directory.

### Monitor idle energy consumption

In [ ]:
from codecarbon import EmissionsTracker, OfflineEmissionsTracker
tracker = EmissionsTracker()
# tracker = OfflineEmissionsTracker(country_iso_code="#")

tracker.start()

time.sleep(10) 

tracker.stop()

Using `time.sleep(10)`, mesure the idle consumption of the GPU, whithout doing any compute. What does it corresponds to ?

In [ ]:
Energy_idle = tracker.final_emissions_data.energy_consumed
time_idle = tracker.final_emissions_data.duration
CO2e_idle = tracker.final_emissions * 1000

print(f"Energy idle : {Energy_idle:.2f}Wh" )
print(f"Duration idle : {time_idle:.2f}s" )
print(f"Emissions idle : {CO2e_idle:.3f}gCO2e")

In [ ]:
P_idle = #
print(f"Power train : {P_idle:.3f}W")

Monitor the emissions from training the digital classifier and compare with the results from green algorithms methodology.

In [ ]:
tracker = OfflineEmissionsTracker(country_iso_code="USA")

tracker.start()

train(model, 1)

tracker.stop()

In [ ]:
Energy_train = tracker.final_emissions_data.energy_consumed
time_train = tracker.final_emissions_data.duration
CO2e_train = tracker.final_emissions * 1000

print(f"Energy train : {Energy_train:.2f}Wh" )
print(f"Duration train : {Energy_train:.2f}s" )
print(f"Emissions train : {CO2e_train:.3f}gCO2e")

In [ ]:
P_train = #
print(f"Power train : {P_train:.3f}W")

# Inference costs


In this section, we want to calculate the energy cost of inference. We will use pretrained models from torchvision available at https://docs.pytorch.org/vision/stable/models.html


Load several models from torchvision.

In [ ]:
from torchvision.models import resnet50, inception_v3, vit_b_16

models = [
    ("resnet50",     resnet50,     224),
    ("inception_v3", inception_v3, 299),
    ("vit_b_16",     vit_b_16,     224),
]

Monitor the consumption of a 1000 forwards pass for each model using different sizes of batch, and compare with the performance.

In [ ]:
for name, model, size in models:
    model = model(weights="DEFAULT").to(device).eval()
    #random input
    x = torch.randn(32, 3, size, size, device=device)

    tracker = OfflineEmissionsTracker(country_iso_code="USA")
    tracker.start()

    model.eval()
    with torch.no_grad():
        for i in range(100):
          model(x)
        
    tracker.stop()

    print(f"{name}")
    Energy_inf = tracker.final_emissions_data.energy_consumed
    time_inf = tracker.final_emissions_data.duration
    CO2e_inf = tracker.final_emissions * 1000

    print(f"Energy inf : {Energy_inf:.2f}Wh" )
    print(f"Duration inf : {Energy_inf:.2f}s" )
    print(f"Emissions inf : {CO2e_inf:.3f}gCO2e")

## Hardware independant metrics

We will use `thop`library to count the number of parameters and operations of a model. All the documentation is available here : https://github.com/Lyken17/pytorch-OpCounter

In [ ]:
!pip install thop > /dev/null

In [ ]:
from thop import profile, clever_format

In [ ]:
for name, model, size in models:
  model = model(weights="DEFAULT").to(device).eval()
  x = torch.randn(32, 3, size, size, device=device)

  macs, params = profile(model, inputs=(x, ))
  macs, params = clever_format([macs, params], "%.3f")
  print(name)
  print(macs,params)

In [ ]:
models = #
energy = #
macs = #
params = #

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# FLOPs vs Duration
axes[0].scatter(energy, macs)
axes[0].set_xlabel('Duration (s)')
axes[0].set_ylabel('GFLOPs')
axes[0].grid(True, alpha=0.3)

# Number of parameters vs Duration
axes[1].scatter(energy, params)
axes[1].set_xlabel('Duration (s)')
axes[1].set_ylabel('Number of parameters')
axes[1].grid(True, alpha=0.3)


plt.show()

# Prompting with your best-LLMs-friends :)

Let’s estimate your energy consumption and carbon footprint of using LLMs last week using
https://calculator.ecologits.ai/

You can download your history
https://help.openai.com/en/articles/7260999-exporting-your-chatgpt-history-and-data?utm_source=chatgpt.com



Location of the datacenter :
https://docs.google.com/spreadsheets/d/1XkPTkrGxpwWpIVIxpVvgRJuInSZsqbndTQbFGcHhdd0/edit?gid=122417664#gid=122417664

| AI Provider | Cloud Provider | Location | PUE | WUE (L/kWh) |
| --- | --- | :-: | :-: | :-: |
| Anthropic | AWS, Google | USA | 1.09 – 1.14 | 0.13 – 1.00 |
| Cohere | Google | USA | 1.09 | 1.00 |
| Google | Google | USA  | 1.09 | 1.00 |
| Hugging Face | AWS, Google | USA | 1.09 – 1.14 | 0.13 – 1.00 |
| Mistral AI | Microsoft | SWE | 1.16 | 0.09 |
| OpenAI | Microsoft | USA | 1.20 | 0.569 |
| Azure OpenAI | Microsoft | USA | 1.20 | 0.569 |

In [ ]:
prompts_per_day = #
energy_prompt = #Wh

Energy_prompting_year = #Wh
CO2e_prompting_year = #gCO2

print(f"Energy LLM: {Energy_prompting_year:.3f} kWh")
print(f"Carbon LLM per week : {CO2e_prompting_year:3f} kgCO2")

Compare this with your consumption with video streaming (Insta / TV / Netflix)
https://impactco2.fr/outils/usagenumerique/streamingvideo

In [ ]:
CO2e_streaming = #

Considering that there are 2,5 billions of prompts everyday on ChatGPT, try to estimate the impact per year.


In [ ]:
prompts_chatGPT_all = #
Energy_chatGPT_all = #
CO2_chatGPT_all = #